# 01 · The GitHub search API

**Phase 1.1: try the search yourself.** Call `GET /search/repositories?q=inventory management&per_page=5` and look at one result.

**Goal:** see exactly what the API gives us for one repository, and find the fields we will keep:
`full_name`, `description`, `topics`, `stargazers_count`, `forks_count`, `language`, `archived`, `fork`, `pushed_at`.

In [1]:
import os

import pandas as pd
import requests
from dotenv import load_dotenv

from discovery.config import BACKEND, SEED, set_seed

load_dotenv(BACKEND / ".env")          # reads GITHUB_TOKEN; never print it
set_seed()
pd.set_option("display.max_colwidth", 80)

API = "https://api.github.com"
session = requests.Session()
session.headers.update({
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}",
})
print("token loaded:", bool(os.environ.get("GITHUB_TOKEN")), "| seed:", SEED)

token loaded: True | seed: 42


## 1. Call the search

In [2]:
res = session.get(f"{API}/search/repositories",
                  params={"q": "inventory management", "per_page": 5})
res.raise_for_status()
data = res.json()

print("status:", res.status_code)
print("total_count:", f"{data['total_count']:,}")
print("incomplete_results:", data["incomplete_results"])
print("search requests left this minute:",
      res.headers["X-RateLimit-Remaining"], "of", res.headers["X-RateLimit-Limit"])
print("results on this page:", len(data["items"]))

status: 200
total_count: 146,525
incomplete_results: False
search requests left this minute: 29 of 30
results on this page: 5


**What it means:** `total_count` is how many repositories match, but the search API only lets us page through the first **1,000**. Search has its own limit of 30 requests per minute, separate from the 5,000 per hour for the rest of the API.

## 2. One result: how big is it?

In [3]:
repo = data["items"][0]
print(len(repo), "fields in one result. All of them:")
print(", ".join(sorted(repo)))

83 fields in one result. All of them:
allow_forking, archive_url, archived, assignees_url, blobs_url, branches_url, clone_url, collaborators_url, comments_url, commits_url, compare_url, contents_url, contributors_url, created_at, default_branch, deployments_url, description, disabled, downloads_url, events_url, fork, forks, forks_count, forks_url, full_name, git_commits_url, git_refs_url, git_tags_url, git_url, has_discussions, has_downloads, has_issues, has_pages, has_projects, has_pull_requests, has_wiki, homepage, hooks_url, html_url, id, is_template, issue_comment_url, issue_events_url, issues_url, keys_url, labels_url, language, languages_url, license, merges_url, milestones_url, mirror_url, name, node_id, notifications_url, open_issues, open_issues_count, owner, permissions, private, pull_request_creation_policy, pulls_url, pushed_at, releases_url, score, size, ssh_url, stargazers_count, stargazers_url, statuses_url, subscribers_url, subscription_url, svn_url, tags_url, teams_url

## 3. The fields we keep

In [4]:
FIELDS = ["full_name", "description", "topics", "stargazers_count", "forks_count",
          "language", "archived", "fork", "pushed_at"]

pd.Series({f: repo[f] for f in FIELDS}, name=repo["full_name"]).to_frame("value")

,value
full_name,inventree/InvenTree
description,Open Source Inventory Management System
topics,"[django, hacktoberfest, python]"
stargazers_count,7890
forks_count,1633
language,Python
archived,False
fork,False
pushed_at,2026-11-22T13:15:40Z


## 4. All five results side by side

In [5]:
df = pd.DataFrame([{f: r[f] for f in FIELDS} for r in data["items"]])
df["topics"] = df["topics"].map(lambda t: ", ".join(t) or "(none)")
df

,full_name,description,topics,stargazers_count,forks_count,language,archived,fork,pushed_at
0,inventree/InvenTree,Open Source Inventory Management System,"django, hacktoberfest, python",7890,1633,Python,False,False,2026-11-22T13:15:40Z
1,beck-source/inventory-management,Factory Inventory Management System — Claude Code workshop baseline app,(none),39,3769,Vue,False,False,2026-09-29T21:50:27Z
2,mauricecalhoun/inventory,Inventory Management for Laravel 4 / 5,(none),431,198,PHP,False,False,2022-10-27T06:52:53Z
3,partkeepr/PartKeepr,Open Source Inventory Management,"inventory, javascript, partkeepr, php",1532,417,JavaScript,True,False,2023-05-30T13:28:03Z
4,ed-roh/inventory-management,NaN,(none),441,260,TypeScript,False,False,2024-08-01T16:41:23Z


## 5. What is *not* in a search result

The README, the full language breakdown and the contributors are **not** here: each needs its own request per repository (`/repos/{owner}/{repo}/readme`, `/languages`, `/contributors`). That is what makes collecting 10,000 repositories slow, and why the crawler caches every response.

In [6]:
missing = [k for k in ("readme", "languages", "contributors") if k in repo]
print("README / languages / contributors in the search result:", missing or "none of them")
print("but the URLs to fetch them are:", repo["languages_url"], repo["contributors_url"], sep="\n  ")

README / languages / contributors in the search result: none of them
but the URLs to fetch them are:
  https://api.github.com/repos/inventree/InvenTree/languages
  https://api.github.com/repos/inventree/InvenTree/contributors


## 6. One repository in full: README, languages, contributors

The search result is only a summary. For the repository above we make three more calls, and after each one print the rate-limit headers:
- `X-RateLimit-Remaining`: how many requests are left in this window;
- `X-RateLimit-Reset`: when the window resets, as a Unix timestamp (shown here as local time too).

In [7]:
from datetime import datetime


def get(path, **params):
    """GET one API path and print the rate-limit headers after the call."""
    r = session.get(f"{API}{path}", params=params)
    r.raise_for_status()
    reset = int(r.headers["X-RateLimit-Reset"])
    print(f"GET {path}  ->  {r.status_code} | "
          f"X-RateLimit-Remaining: {r.headers['X-RateLimit-Remaining']} of {r.headers['X-RateLimit-Limit']} | "
          f"X-RateLimit-Reset: {reset} ({datetime.fromtimestamp(reset):%H:%M:%S}) | "
          f"resource: {r.headers.get('X-RateLimit-Resource')}")
    return r.json()


owner, name = repo["full_name"].split("/")
print("repository:", repo["full_name"])

repository: inventree/InvenTree


### 6.1 README: Base64 in `content`, decode it

In [8]:
import base64

readme = get(f"/repos/{owner}/{name}/readme")
print("file:", readme["path"], "| encoding:", readme["encoding"], "| size:", readme["size"], "bytes")
print("raw content starts:", readme["content"][:60].replace("\n", ""), "...")

text = base64.b64decode(readme["content"]).decode("utf-8", errors="replace")
print(f"decoded: {len(text):,} characters, {text.count(chr(10)) + 1} lines\n")
print(text)

GET /repos/inventree/InvenTree/readme  ->  200 | X-RateLimit-Remaining: 4989 of 5000 | X-RateLimit-Reset: 1795363207 (23:00:07) | resource: core
file: README.md | encoding: base64 | size: 12649 bytes
raw content starts: PGRpdiBhbGlnbj0iY2VudGVyIj4KICA8aW1nIHNyYz0iYXNzZXRzL2ltYWdl ...
decoded: 12,642 characters, 213 lines

<div align="center">
  <img src="assets/images/logo/inventree.png" alt="InvenTree logo" width="200" height="auto" />
  <h1>InvenTree</h1>
  <p>Open Source Inventory Management System </p>

<!-- Badges -->
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/license/MIT)![GitHub tag (latest SemVer)](https://img.shields.io/github/v/tag/inventree/inventree)[![Artifact Hub](https://img.shields.io/endpoint?url=https://artifacthub.io/badge/repository/inventree)](https://artifacthub.io/packages/search?repo=inventree)
![CI](https://github.com/inventree/inventree/actions/workflows/qc_checks.yaml/badge.svg)
[![Documentation Status](https

**What it means:** the README is plain Markdown, but full of HTML tags, badges, image links and code blocks. None of that describes what the project *does*, which is why cleaning (Phase 4) strips it before we index the text.

### 6.2 Languages: bytes of code per language

In [9]:
langs = get(f"/repos/{owner}/{name}/languages")
total = sum(langs.values())
pd.DataFrame({"bytes": langs, "share": {k: f"{v / total:.1%}" for k, v in langs.items()}})

GET /repos/inventree/InvenTree/languages  ->  200 | X-RateLimit-Remaining: 4988 of 5000 | X-RateLimit-Reset: 1795363207 (23:00:07) | resource: core


,bytes,share
Python,7053435,69.3%
TypeScript,3006912,29.5%
HTML,59885,0.6%
Shell,39271,0.4%
JavaScript,8648,0.1%
Dockerfile,8407,0.1%
CSS,3494,0.0%
Procfile,426,0.0%


**What it means:** the search result's `language` field is only the *largest* one. `/languages` shows the whole stack, so a Django + JavaScript project can also match a JavaScript query.

### 6.3 Contributors

In [10]:
contributors = get(f"/repos/{owner}/{name}/contributors", per_page=100)
print(len(contributors), "contributors on the first page (max 100 per page)\n")
pd.DataFrame(contributors)[["login", "type", "contributions"]].head(10)

GET /repos/inventree/InvenTree/contributors  ->  200 | X-RateLimit-Remaining: 4987 of 5000 | X-RateLimit-Reset: 1795363207 (23:00:07) | resource: core
100 contributors on the first page (max 100 per page)



,login,type,contributions
0,SchrodingersGat,User,13210
1,matmair,User,3352
2,dependabot[bot],Bot,502
3,github-actions[bot],Bot,443
4,eeintech,User,350
5,wolflu05,User,61
6,1337joe,User,33
7,martonmiklos,User,33
8,SergeoLacruz,User,29
9,LavissaWoW,User,27


**What it means:** contributors link repositories together. Two repositories that share contributors are related, which becomes an edge in the repository graph for PageRank (Phase 12). Bots (`type: Bot`) should be dropped before that.

### 6.4 The cost per repository

Every call above used the **core** limit (5,000 per hour), not the search limit (30 per minute). Collecting one repository in full costs about **3 core requests** (README, languages, contributors) on top of its share of a search page, so 10,000 repositories is about 30,000 requests: around **6 hours** of the hourly limit. That's why the crawler must cache every response in `data/raw/` and be able to resume.

In [11]:
limits = session.get(f"{API}/rate_limit").json()["resources"]
for k in ("core", "search"):
    left, lim = limits[k]["remaining"], limits[k]["limit"]
    print(f"{k:6}: {left:,} of {lim:,} left, resets at {datetime.fromtimestamp(limits[k]['reset']):%H:%M:%S}")

core  : 4,987 of 5,000 left, resets at 23:00:08
search: 29 of 30 left, resets at 22:27:04


## 7. Learn the limits (Phase 1.2)

| Limit | Value | What it means for us |
|---|---|---|
| Search requests | 30 per minute | Wait at least 2 s between search pages |
| Other requests | 5,000 per hour | README + languages + contributors = 3 calls per repository |
| Results per search | 1,000 | Split big searches into smaller slices (Phase 2) |
| Page size | 100 | 10 pages per search at most |

We don't take these on trust: each one is checked below, spending only a few requests.

### 7.1 Both limits, from `/rate_limit` (this call is free)

In [12]:
import time

limits = session.get(f"{API}/rate_limit").json()["resources"]
pd.DataFrame({
    k: {"limit": limits[k]["limit"], "remaining": limits[k]["remaining"],
        "resets at": f"{datetime.fromtimestamp(limits[k]['reset']):%H:%M:%S}",
        "window": "1 hour" if k == "core" else "1 minute"}
    for k in ("core", "search")
}).T

,limit,remaining,resets at,window
core,5000,4987,23:00:08,1 hour
search,30,29,22:27:04,1 minute


### 7.2 A polite search helper: at least 2 s between search requests

30 per minute is one every 2 s. The helper waits if the last search was less than 2 s ago, so a long crawl never hits the limit.

In [13]:
SEARCH_GAP = 60 / limits["search"]["limit"]   # 2.0 s
_last_search = 0.0


def search(q, page=1, per_page=100):
    """One search page, never faster than one every SEARCH_GAP seconds."""
    global _last_search
    wait = SEARCH_GAP - (time.monotonic() - _last_search)
    if wait > 0:
        time.sleep(wait)
    _last_search = time.monotonic()
    return session.get(f"{API}/search/repositories", params={"q": q, "page": page, "per_page": per_page})


t0 = time.monotonic()
for page in (1, 2, 3):
    r = search("inventory management", page=page, per_page=5)
    print(f"page {page}: {r.status_code} at {time.monotonic() - t0:4.1f} s | "
          f"search left this minute: {r.headers['X-RateLimit-Remaining']}")

page 1: 200 at  0.6 s | search left this minute: 28


page 2: 200 at  2.6 s | search left this minute: 27


page 3: 200 at  4.6 s | search left this minute: 26


### 7.3 Page size: ask for 150, get 100

In [14]:
r = search("inventory management", per_page=150)
print("asked for 150 per page, got", len(r.json()["items"]))

asked for 150 per page, got 100


### 7.4 Results per search: page 11 of 100 is past the 1,000th result

In [15]:
for page in (10, 11):
    r = search("inventory management", page=page, per_page=100)
    body = r.json()
    print(f"page {page}: {r.status_code}", f"{len(body['items'])} items" if r.ok else body.get("message"))

page 10: 200 100 items


page 11: 422 Only the first 1000 search results are available


**What it means:** page 10 still answers (results 901–1,000); page 11 is refused, however large `total_count` is. The only way to reach more repositories is more, narrower searches (Phase 2).

### 7.5 How the query changes `total_count`

The searches from the guide's table, re-run now. The full sentence a user might type finds **nothing**: GitHub's search needs every word to match. That gap is the reason this project exists.

In [16]:
QUERIES = [
    "Find a beginner-friendly inventory management system using NestJS and PostgreSQL",
    "inventory management nestjs postgresql",
    "inventory management",
    "inventory management stars:>=5",
    "inventory management django",
    "inventory management nestjs",
    "point of sale react",
    "school management laravel",
]
counts = pd.DataFrame([{"query": q, "total_count": search(q, per_page=1).json()["total_count"]} for q in QUERIES])
counts["fits in one search (≤ 1,000)"] = counts["total_count"] <= 1000
counts.style.format({"total_count": "{:,}"})

,query,total_count,"fits in one search (≤ 1,000)"
0,Find a beginner-friendly inventory management system using NestJS and PostgreSQL,0,True
1,inventory management nestjs postgresql,151,True
2,inventory management,"146,526",False
3,inventory management stars:>=5,"1,360",False
4,inventory management django,"2,597",False
5,inventory management nestjs,303,True
6,point of sale react,"1,097",False
7,school management laravel,"1,516",False


In [17]:
share = counts.set_index("query").loc["inventory management stars:>=5", "total_count"] / \
        counts.set_index("query").loc["inventory management", "total_count"]
print(f"a floor of 5 stars keeps {share:.1%} of 'inventory management'")

a floor of 5 stars keeps 0.9% of 'inventory management'


### 7.6 The budget

In [18]:
N_REPOS, CALLS_PER_REPO = 10_000, 3                  # README + languages + contributors
SLICES, PAGES = 12 * 12, 10                           # domains × frameworks, at most 10 pages each

core_calls = N_REPOS * CALLS_PER_REPO
search_calls = SLICES * PAGES
print(f"core:   {core_calls:,} calls / {limits['core']['limit']:,} per hour = {core_calls / limits['core']['limit']:.1f} hours")
print(f"search: {search_calls:,} calls at one every {SEARCH_GAP:.0f} s   = {search_calls * SEARCH_GAP / 60:.0f} minutes (at most)")

core:   30,000 calls / 5,000 per hour = 6.0 hours
search: 1,440 calls at one every 2 s   = 48 minutes (at most)


**Key idea:** the README, languages and contributors are the expensive part, not the search. Plan to run the crawl over a night, save every response to `data/raw/` as it arrives, and make it **resumable**: a rerun skips what is already saved.

## What we found

- **146,513** repositories match "inventory management", but search only lets us page through the first **1,000**. We have to slice the query (domain × framework × star range) to reach more.
- One result has **83 fields**; we keep **9**. Most of the rest are URLs to other endpoints.
- **`topics` is empty for 3 of 5** results, so we can't rely on topics alone to detect technologies: the README and `language` have to fill the gap.
- **`description` can be missing** (`ed-roh/inventory-management`), so the README is the main text we index.
- **`archived: True`** (`partkeepr/PartKeepr`): still readable, but no longer maintained. Decide in cleaning whether to drop or down-rank it.
- **Forks can be far above stars** (`beck-source/inventory-management`: 39 stars, 3,769 forks, a workshop template people fork). That's a warning for the popularity score: forks alone can be misleading.
- **`pushed_at` ranges from 2022 to today**: useful for a "still active" signal or filter.
- The README, full language breakdown and contributors are **not** in the search result. Each needs one more request per repository.
- **README** comes back Base64-encoded in `content`; decoded, InvenTree's is 12,642 characters and starts with HTML, a logo and a row of badges. Cleaning has to strip all of that.
- **`language` hides the stack:** it says Python, but `/languages` shows **29.5% TypeScript** (the React frontend). Technology detection should use `/languages`, not just `language`.
- **Contributors include bots:** 2 of the top 4 are `dependabot[bot]` and `github-actions[bot]`. Drop `type: Bot` before building the graph. The first page stops at 100, so big projects need paging, or we keep only the top contributors.
- **Rate limits:** each call used 1 of the **5,000 core** requests (4,998 → 4,997 → 4,996), separate from the **30 search** requests per minute. About 3 core calls per repository means ~30,000 for 10,000 repositories, so the crawler must cache and resume.

**Limits (7):**
- **Search: 30 per minute**, so the search helper waits 2 s between requests (pages came back at 0.6, 2.6, 4.6 s). **Core: 5,000 per hour.** Both read for free from `/rate_limit`.
- **Page size is capped at 100:** asking for 150 returns 100, without an error.
- **The 1,000 cap is enforced:** page 10 of 100 works, page 11 returns **422** "Only the first 1000 search results are available".
- **The full sentence a user would type finds 0 repositories**; "inventory management" finds 146,526. A 5-star floor keeps 1,360 (0.9%).
- **Budget:** 30,000 core calls ≈ 6 hours; 1,440 search calls ≈ 48 minutes at most. The core calls are the slow part, so the crawl runs overnight and must resume.